# SMS Spam Modeling Pipeline

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark = SparkSession.builder.appName('sms-spam-modeling').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

clean_df = spark.read.parquet('/content/artifacts/clean_sms.parquet')
print('Rows:', clean_df.count())
print('Columns:', clean_df.columns)
clean_df.show(5, truncate=False)

Rows: 5572
Columns: ['label', 'label_text', 'message', 'msg_len_chars', 'msg_len_words']
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|label|label_text|message                                                                                                                                                    |msg_len_chars|msg_len_words|
+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-------------+
|0    |ham       |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                            |111          |20           |
|0    |ham       |Ok lar... Joking wif u oni...                                                    

In [3]:
train_df, test_df = clean_df.select('label', 'message').randomSplit([0.8, 0.2], seed=42)
print('Train rows:', train_df.count())
print('Test rows:', test_df.count())

tokenizer = Tokenizer(inputCol='message', outputCol='tokens')
stopwords = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
tf = HashingTF(inputCol='filtered_tokens', outputCol='raw_features', numFeatures=2**16)
idf = IDF(inputCol='raw_features', outputCol='features')
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=30)

pipeline = Pipeline(stages=[tokenizer, stopwords, tf, idf, lr])
print('Pipeline ready')

Train rows: 4501
Test rows: 1071
Pipeline ready


In [4]:
model = pipeline.fit(train_df)
pred_df = model.transform(test_df)

pred_df.select('label', 'prediction', 'probability', 'message').show(10, truncate=False)

+-----+----------+-------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|probability                                |message                                                                                                                                                                                                                                                                                                                                                                                         |
+-----+----------+-------------------------------------------+------------------------------------------

In [5]:
tp = pred_df.filter((F.col('label') == 1) & (F.col('prediction') == 1)).count()
tn = pred_df.filter((F.col('label') == 0) & (F.col('prediction') == 0)).count()
fp = pred_df.filter((F.col('label') == 0) & (F.col('prediction') == 1)).count()
fn = pred_df.filter((F.col('label') == 1) & (F.col('prediction') == 0)).count()

total = tp + tn + fp + fn
accuracy = (tp + tn) / total
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')

TP=130, TN=914, FP=9, FN=18
Accuracy : 0.9748
Precision: 0.9353
Recall   : 0.8784
F1-score : 0.9059


In [6]:
from pyspark.ml import PipelineModel

model_path = '/content/artifacts/models/sms_spam_pipeline'
model.write().overwrite().save(model_path)
print('Saved model to:', model_path)

loaded_model = PipelineModel.load(model_path)

new_messages = spark.createDataFrame(
    [
        ('Congratulations! You won a free ticket. Reply WIN now!',),
        ('Can we meet tomorrow at 10 to finish the report?',),
        ('URGENT: claim your reward by calling this number now',),
    ],
    ['message']
 )

new_pred = loaded_model.transform(new_messages)
new_pred.select('message', 'prediction', 'probability').show(truncate=False)

Saved model to: /content/artifacts/models/sms_spam_pipeline
+------------------------------------------------------+----------+------------------------------------------+
|message                                               |prediction|probability                               |
+------------------------------------------------------+----------+------------------------------------------+
|Congratulations! You won a free ticket. Reply WIN now!|1.0       |[1.123837476691011E-6,0.9999988761625233] |
|Can we meet tomorrow at 10 to finish the report?      |0.0       |[0.9999999992745252,7.254747913520987E-10]|
|URGENT: claim your reward by calling this number now  |0.0       |[0.9892299285182212,0.010770071481778776] |
+------------------------------------------------------+----------+------------------------------------------+

